In [ ]:
import os
import subprocess
from Bio.PDB import PDBParser, Select, PDBIO
from loguru import logger
import sys
logger.remove()

logger.add(sys.stdout, format="{message}", level="INFO")

def extract_sdf_from_complex(input_pdb_file, output_dir='./test'):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    filename = os.path.basename(input_pdb_file)
    filename_no_ext = os.path.splitext(filename)[0]
    parts = filename_no_ext.split('_')
    pdb_id = parts[0]
    bioml_number = parts[2]
    ligands_index = parts.index("chains") - 1
    ligands = parts[3:ligands_index + 1]
    chains = parts[ligands_index + 2].split('_')

    logger.debug(f"PDB ID: {pdb_id}")
    logger.debug(f"Biomolecule number: {bioml_number}")
    logger.debug(f"Ligands: {ligands}")
    logger.debug(f"Chains: {chains}")

    class LigandSelect(Select):
        def __init__(self, resname):
            self.resname = resname

        def accept_residue(self, residue):
            return residue.get_resname() == self.resname

    def extract_ligands(pdb_file, output_dir, ligands):
        parser = PDBParser(QUIET=True)
        structure_id = os.path.basename(pdb_file).split('.')[0]
        try:
            structure = parser.get_structure(structure_id, pdb_file)
        except Exception as e:
            logger.debug(f"Error parsing file {pdb_file}: {e}")
            return

        io = PDBIO()
        io.set_structure(structure)
        
        for ligand in ligands:
            select = LigandSelect(resname=ligand)
            output_file = os.path.join(output_dir, f"{ligand}.pdb")
            io.save(output_file, select)
            logger.debug(f"Ligand {ligand} saved to file {output_file}")
        return output_dir

    ligand_files_dir = extract_ligands(input_pdb_file, output_dir, ligands)

    def convert_pdb_to_sdf(input_dir, output_dir):
        pdb_files = [f for f in os.listdir(input_dir) if f.endswith('.pdb')]
        sdf_files = []
        
        for pdb_file in pdb_files:
            input_file = os.path.join(input_dir, pdb_file)
            output_file = os.path.join(output_dir, pdb_file.replace('.pdb', '.sdf'))
            
            command = [
                "obabel",
                input_file,
                "-O", output_file,
                "-h"
            ]
            
            try:
                result = subprocess.run(command, check=True, capture_output=True, text=True)
                sdf_files.append(output_file)
                logger.debug(result.stdout)
            except subprocess.CalledProcessError as e:
                logger.debug(f"Error converting file {input_file}: {e.stderr}")
        
        return sdf_files

    ligand_sdf_files = convert_pdb_to_sdf(ligand_files_dir, output_dir)
    return ligand_sdf_files

list_sdf = extract_sdf_from_complex('../lpce/tests/separated/1a05_bioml_1_IPM_MG_chains_A_B_processed.pdb')
list_sdf

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdMolTransforms
from loguru import logger

def load_sdf_files(file_list):
    mols = []
    for file_path in file_list:
        mol_supplier = Chem.SDMolSupplier(file_path)
        mols.extend([mol for mol in mol_supplier if mol is not None])
        logger.debug(f"Loaded molecules from {file_path}")
    return mols

def detect_and_remove_clashes(mol, clash_threshold=0.6):
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    conf = mol.GetConformer()
    clashes = []
    for i in range(mol.GetNumAtoms()):
        for j in range(i + 1, mol.GetNumAtoms()):
            dist = rdMolTransforms.GetBondLength(conf, i, j)
            if dist < clash_threshold:
                clashes.append((i, j, dist))
                logger.info(f"Clash detected: Atoms {i}-{j} with distance {dist:.3f} Å")

    for i, j, dist in clashes:
        if mol.GetAtomWithIdx(i).GetAtomicNum() == 1:
            mol.GetAtomWithIdx(i).SetNoImplicit(True)
            mol.RemoveAtom(i)
            logger.debug(f"Removed hydrogen atom at index {i} due to clash with atom {j}")
        elif mol.GetAtomWithIdx(j).GetAtomicNum() == 1:
            mol.GetAtomWithIdx(j).SetNoImplicit(True)
            mol.RemoveAtom(j)
            logger.debug(f"Removed hydrogen atom at index {j} due to clash with atom {i}")
        else:
            logger.debug(f"Unable to resolve clash between heavy atoms {i}-{j}; manual review needed")

    return mol

def remove_clashes(file_list):
    mols = load_sdf_files(file_list)
    processed_mols = []
    for mol in mols:
        processed_mol = detect_and_remove_clashes(mol)
        processed_mols.append(processed_mol)
        logger.debug("Molecule processed for clashes")
    return processed_mols

list_sdf = ['./test/MG.sdf', './test/IPM.sdf']
processed_molecules = remove_clashes(list_sdf)


In [ ]:
import subprocess
from pathlib import Path

def run_xtb_charge_analysis(sdf_files):
    results = {}
    
    for sdf_path in sdf_files:
        sdf_file = Path(sdf_path)
        
        if not sdf_file.exists():
            print(f"Файл {sdf_path} не найден.")
            continue
        
        # Конвертируем SDF в XYZ с помощью OpenBabel
        xyz_file = sdf_file.with_suffix('.xyz')
        subprocess.run(['obabel', str(sdf_file), '-O', str(xyz_file)], capture_output=True)
        
        # Запускаем xTB для оценки зарядов
        xtb_output = subprocess.run(['xtb', str(xyz_file), '--charges', '--wbo'], capture_output=True, text=True)
        
        # Проверка успешности выполнения xTB
        if xtb_output.returncode != 0:
            print(f"Ошибка при запуске xTB для файла {sdf_file.name}")
            continue
        
        # Считываем данные из файла charges
        charges_file = Path('charges')
        if charges_file.exists():
            with charges_file.open() as f:
                charges = [float(line.strip()) for line in f.readlines()]
        else:
            print(f"Файл 'charges' не найден для {sdf_file.name}")
            charges = []

        # Считываем данные из файла wbo
        wbo_file = Path('wbo')
        wbo_data = []
        if wbo_file.exists():
            with wbo_file.open() as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 3:
                        atom1, atom2, bond_order = int(parts[0]), int(parts[1]), float(parts[2])
                        wbo_data.append((atom1, atom2, bond_order))
        else:
            print(f"Файл 'wbo' не найден для {sdf_file.name}")

        # Сохраняем результаты
        results[sdf_file.name] = {
            "Mulliken Charges": charges,
            "Wiberg Bond Orders": wbo_data
        }
        
        # Удаление временных файлов, если необходимо
        charges_file.unlink(missing_ok=True)
        wbo_file.unlink(missing_ok=True)
        xyz_file.unlink(missing_ok=True)
    
    return results

list_sdf = ['./test/MG.sdf', './test/IPM.sdf']
charge_analysis_results = run_xtb_charge_analysis(list_sdf)
charge_analysis_results

def check_and_correct_total_charge(charge_analysis_results, target_charge=0):
    corrected_charges = {}

    for filename, data in charge_analysis_results.items():
        mulliken_charges = data.get("Mulliken Charges", [])
        
        # Суммируем заряды, чтобы проверить общую заряженность
        total_charge = sum(mulliken_charges)
        charge_deviation = total_charge - target_charge
        
        # Проверяем отклонение от целевого заряда
        if abs(charge_deviation) > 1e-5:
            logger.info(f"Корректировка зарядов для файла {filename}")
            logger.info(f"Текущая суммарная заряженность: {total_charge}, Ожидаемая заряженность: {target_charge}")
            
            # Равномерно распределяем корректировку по каждому атому
            correction_per_atom = -charge_deviation / len(mulliken_charges)
            corrected_charges[filename] = [charge + correction_per_atom for charge in mulliken_charges]
            
            logger.info(f"Скорректированные заряды для файла {filename}: {corrected_charges[filename]}")
        else:
            # Если корректировка не нужна, сохраняем исходные заряды
            corrected_charges[filename] = mulliken_charges
            logger.info(f"Заряд молекулы {filename} уже корректен.")

    return corrected_charges

corrected_charges = check_and_correct_total_charge(charge_analysis_results)
print("Corrected charges:", corrected_charges)
